# Walkthrough · the New Testament scholar: gold Koine, honest machines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/walkthrough-nt-koine.ipynb)

*You read Koine: the Greek New Testament with its curated annotations, and the occasional text nobody has annotated for you.* This notebook is one working session, following [Recipes → workflow D](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes) (the New Testament scholar).

By the end you will have:

* gold per-token annotations (lemma, Robinson morphology, Strong's number, gloss) on real verses;
* offline dictionary glosses from the bundled Dodson lexicon;
* a lemma-based concordance built in a few lines;
* the automatic pipeline run where no gold exists, with its **evidence classes** saying which analyses to trust;
* a review CSV exported, corrected, and applied back: the human-in-the-loop round trip.

**Offline-first.** The bundled sample carries John 1 and Philemon with full gold annotations, and this notebook keeps to that coverage, so every ungated cell runs with no network. (With network access, `load_nt` fetches all 27 books once, ~16 MB, and the same cells run against the full corpus.) The one `RUN_HEAVY` cell is the neural pipeline: the `[neural]` extra plus a ~173 MB model download.

In [ ]:
# pyaegean works fully offline once installed. This installs it the first time
# you open the notebook in Colab; locally it's a no-op if it's already present.
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__)

In [ ]:
# ── The one switch for the optional, downloading cells ──────────────────────
# Leave this False and everything below runs offline against the bundled
# John 1 + Philemon sample. Flip it to True (on decent wifi) for:
#   * greek.use_neural_pipeline()   ~173 MB one-time model download; needs the
#                                   [neural] extra (pip install "pyaegean[neural]")
# It fetches once, then caches. Every other cell runs offline with no downloads.
RUN_HEAVY = False

print("Heavy/optional cells are",
      "ON — first run downloads." if RUN_HEAVY
      else "OFF — the offline core still runs in full.")

## Step 1 · Gold annotations, verse by verse

The NT corpus is Nestle 1904: the morphology, lemmas, and Strong's numbers are CC0, the base text public domain. Every token carries a gold lemma, a Robinson morphology tag, a Strong's number, a reconciled UD part of speech, a short gloss, and its canonical reference. Nothing here is machine-predicted; this is the curated scholarly layer.

In [ ]:
from aegean import greek

jn = greek.load_nt("John", ref="1")     # John 1: in the offline sample AND the full fetch
doc = jn.documents[0]
print(doc.id, "·", len(doc.tokens), "tokens")

for t in [t for t in doc.tokens if t.annotations["ref"] == "John.1.1"][:6]:
    a = t.annotations
    print(f"  {t.text:8} {a['lemma']:6} {a['upos']:6} {a['morph']:9} Strong's {a['strongs']:>4}  {a['gloss'][:28]}")
# John 1 · 828 tokens
#   Ἐν       ἐν     ADP    PREP      Strong's 1722  in, on, among
#   ἀρχῇ     ἀρχή   NOUN   N-DSF     Strong's  746  ruler, beginning
#   ἦν       εἰμί   VERB   V-IAI-3S  Strong's 1510  I am, exist
#   ὁ        ὁ      DET    T-NSM     Strong's 3588  the
#   Λόγος,   λόγος  NOUN   N-NSM     Strong's 3056  a word, speech, divine utter
#   καὶ      καί    CCONJ  CONJ      Strong's 2532  and, even, also, namely

## Step 2 · Morphology you can compute on

Robinson tags are compact (`V-IAI-3S`: verb, imperfect active indicative, third singular) and, because they are data rather than footnotes, they support grammatical concordances directly. Two one-liners: the part-of-speech profile of the chapter, and every imperfect third singular in it.

In [ ]:
from collections import Counter

print("UPOS in John 1:", Counter(t.annotations["upos"] for t in doc.tokens
                                 if t.annotations.get("upos")).most_common(5))

imperfects = [t for t in doc.tokens if t.annotations.get("morph") == "V-IAI-3S"]
print(len(imperfects), "imperfect indicative 3rd-singulars:",
      dict(Counter(t.annotations["lemma"] for t in imperfects)))
# UPOS in John 1: [('VERB', 201), ('NOUN', 158), ('PRON', 126), ('DET', 109), ('CCONJ', 88)]
# 17 imperfect indicative 3rd-singulars: {'εἰμί': 16, 'φημί': 1}
# John 1's famous imperfects (ἦν …) are 16 forms of εἰμί plus one ἔφη.

## Step 3 · Offline glosses, and a concordance

The bundled Dodson lexicon (CC0) glosses any Koine lemma with no download at all. And once every token carries a gold lemma, a concordance is a loop: here is φῶς (light) through John 1, matched by lemma so the genitive φωτός is found too.

In [ ]:
greek.use_dodson()                       # bundled Koine lexicon, zero network
for w in ("λόγος", "χάρις", "μονογενής"):
    print(f"  {w:10} → {greek.gloss_nt(w)}")
#   λόγος      → a word, speech, divine utterance, analogy
#   χάρις      → grace, favor, kindness
#   μονογενής  → only, only-begotten, unique

In [ ]:
toks = doc.tokens
for i, t in enumerate(toks):
    if t.annotations.get("lemma") == "φῶς":
        left = " ".join(x.text for x in toks[max(0, i - 3):i])
        right = " ".join(x.text for x in toks[i + 1:i + 4])
        print(f"{t.annotations['ref']:11} {left:>26}  {t.text}  {right}")
# John.1.4                     ζωὴ ἦν τὸ  φῶς  τῶν ἀνθρώπων. καὶ
# John.1.5              ἀνθρώπων. καὶ τὸ  φῶς  ἐν τῇ σκοτίᾳ
# John.1.7            μαρτυρήσῃ περὶ τοῦ  φωτός,  ἵνα πάντες πιστεύσωσιν
# John.1.8                 ἦν ἐκεῖνος τὸ  φῶς,  ἀλλ’ ἵνα μαρτυρήσῃ
# John.1.8            μαρτυρήσῃ περὶ τοῦ  φωτός.  Ἦν τὸ φῶς
# John.1.9                  φωτός. Ἦν τὸ  φῶς  τὸ ἀληθινὸν, ὃ

## Step 4 · Where there is no gold: the honest pipeline

The NT has gold annotations; your seminar handout, papyrus line, or student composition does not. `greek.pipeline` is the zero-dependency baseline for such text, and it is deliberately honest about itself: each record carries a `lemma_source` **evidence class** (attested / seed / rule / identity / unresolved …), and `greek.needs_review(source)` flags the classes that are guesses. Rather than fabricate a plausible-looking lemma, the baseline returns the surface form and says `unresolved`.

In [ ]:
VERSE = "Ἐν ἀρχῇ ἦν ὁ Λόγος, καὶ ὁ Λόγος ἦν πρὸς τὸν Θεόν, καὶ Θεὸς ἦν ὁ Λόγος."

for r in greek.pipeline(VERSE)[:7]:
    print(f"  {r.text:8} {r.upos:6} {r.lemma:8} {r.lemma_source.value:11}"
          f" needs_review={greek.needs_review(r.lemma_source)}")
#   Ἐν       ADP    ἐν       seed        needs_review=False
#   ἀρχῇ     NOUN   ἀρχή     seed        needs_review=False
#   ἦν       VERB   εἰμί     seed        needs_review=False
#   ὁ        DET    ὁ        seed        needs_review=False
#   Λόγος    NOUN   λόγος    seed        needs_review=False
#   ,        PUNCT  ,        punct       needs_review=False
#   καὶ      CCONJ  καί      seed        needs_review=False

In [ ]:
# How honest is the flag? Score the offline baseline against John 1's gold lemmas,
# split by whether the evidence class asked for review:
gold = [(t.text, t.annotations["lemma"]) for t in doc.tokens if t.annotations.get("lemma")]

conf = conf_right = flag = flag_right = 0
for w, lem in gold:
    pred, src = greek.lemmatize_sourced(w)
    if greek.needs_review(src):
        flag, flag_right = flag + 1, flag_right + (pred == lem)
    else:
        conf, conf_right = conf + 1, conf_right + (pred == lem)

print(f"confident classes: {conf_right}/{conf} correct ({conf_right / conf:.0%})")
print(f"flagged for review: {flag_right}/{flag} correct ({flag_right / flag:.0%})")
# confident classes: 407/441 correct (92%)
# flagged for review: 84/387 correct (22%)
# The flag concentrates the errors where it says they are. Note the honest fine
# print: "confident" is mostly right, not infallible; the review loop below is
# how the residue gets caught.

## Step 5 · The neural pipeline (optional download)

The joint neural pipeline (`greek.use_neural_pipeline()`, the `[neural]` extra) tags, lemmatizes, assigns full morphology, and parses in one pass, and is the most accurate tier pyaegean ships. The measured numbers per treebank, including the out-of-domain NT scores, live in [Benchmarks](https://github.com/ryanpavlicek/pyaegean/wiki/Benchmarks); no accuracy claims are made here that the protocol has not measured. The NT is out of domain for a model trained on literary treebanks: another reason the gold layer above stays the reference.

In [ ]:
if not RUN_HEAVY:
    print('[skipped] set RUN_HEAVY = True (with pip install "pyaegean[neural]") for the neural pipeline.')
else:
    greek.use_neural_pipeline()          # one-time ~173 MB model fetch, then cached
    for r in greek.pipeline(VERSE, parse=True)[:5]:
        print(f"  {r.text:8} {r.upos:6} {r.lemma:6} head={r.head} {r.relation:6} {r.feats}")
    greek.disable_neural_pipeline()      # back to the offline baseline
    #   Ἐν       ADP    ἐν     head=2 case   _
    #   ἀρχῇ     NOUN   ἀρχή   head=0 root   Case=Dat|Gender=Fem|Number=Sing
    #   ἦν       VERB   εἰμί   head=2 cop    Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin|Voice=Act
    #   ὁ        DET    ὁ      head=5 det    Case=Nom|Gender=Masc|Number=Sing
    #   Λόγος    NOUN   λόγος  head=3 nsubj  Case=Nom|Gender=Masc|Number=Sing
    # One call: UPOS, lemma, full UD morphology, and a dependency parse.

## Step 6 · The review loop: export, correct, apply

Automatic analysis does not end the workflow; a scholar's correction does. The loop: `annotate_corpus` fills annotations on your own text, `to_review_table` writes a spreadsheet-safe CSV (UTF-8 BOM, one row per word, formula cells neutralized), a reviewer fills the `correct_*` columns, and `from_review_table` applies the corrections back, keeping the machine value under `lemma__pred` and stamping the reviewer and a provenance note.

Here the "reviewer edits" happen in code so the notebook runs unattended; in practice you open the CSV in any spreadsheet.

In [ ]:
from aegean import io

text = "Χάρις ὑμῖν καὶ εἰρήνη ἀπὸ Θεοῦ Πατρὸς ἡμῶν καὶ Κυρίου Ἰησοῦ Χριστοῦ."   # Phlm 3
own = io.from_text(text, script_id="greek", doc_id="phm-1.3")
own = greek.annotate_corpus(own)        # offline baseline fills lemma/upos/evidence

for t in own.documents[0].tokens:
    a = t.annotations
    if a.get("lemma"):
        print(f"  {t.text:10} {a['lemma']:10} {a['upos']:6} {a['lemma_source']:11} known={a['lemma_known']}")
#   Χάρις      Χάρις      NOUN   unresolved  known=false  ← honest miss (verse-initial capital)
#   ὑμῖν       σύ         NOUN   seed        known=true
#   …
#   Κυρίου     κύριος     NOUN   seed        known=true   ← the curated seed layer resolves
#   Ἰησοῦ      Ἰησοῦς     NOUN   seed        known=true     forms the suffix rules alone
#   Χριστοῦ    Χριστός    NOUN   rule        known=true     could not handle safely

In [ ]:
import csv
import os
import tempfile

review_csv = os.path.join(tempfile.mkdtemp(), "review-phm.csv")   # use your own path
n = io.to_review_table(own, review_csv)
print(n, "rows written")

rows = list(csv.DictReader(open(review_csv, encoding="utf-8-sig")))
for r in rows:
    if r["needs_review"] == "yes":
        print(f"  flagged: {r['token']:10} pred={r['pred_lemma']:10} class={r['evidence_class']}")
# 12 rows written
#   flagged: Χάρις      pred=Χάρις      class=unresolved
#   flagged: εἰρήνη     pred=εἰρήνη     class=unresolved
#   flagged: Πατρὸς     pred=Πατρὸς     class=unresolved
# One of the three flags is already the right lemma (εἰρήνη is its own lemma):
# the flag means "verify me", not "wrong". The other two are real misses. And
# the confident rows measured ~92% correct above, not 100%, so spot-check
# them too when the text matters.

In [ ]:
# The reviewer's pass (in a spreadsheet, normally): fix the two flags that
# were real misses.
for r in rows:
    if r["token"] == "Χάρις":
        r["correct_lemma"] = "χάρις"
        r["reviewer_note"] = "verse-initial capital; the lemma is lowercase"
    if r["token"] == "Πατρὸς":
        r["correct_lemma"] = "πατήρ"
with open(review_csv, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

reviewed = io.from_review_table(review_csv, own, reviewer="you")
for t in reviewed.documents[0].tokens:
    a = t.annotations
    if a.get("review_status"):
        print(f"  {t.text:8} → {a['lemma']:8} (machine said {a['lemma__pred']}; note: {a.get('review_note', '')})")
print([note for note in reviewed.provenance.notes if note.startswith("review")])
#   Χάρις    → χάρις    (machine said Χάρις; note: verse-initial capital; the lemma is lowercase)
#   Πατρὸς   → πατήρ    (machine said Πατρὸς; note: )
# ['review: 2 tokens corrected by you (2026-07-11)']
# The corrected corpus keeps the machine value under lemma__pred, so the
# correction itself stays auditable.

## Where to go next

* [New Testament](https://github.com/ryanpavlicek/pyaegean/wiki/New-Testament): the corpus, addressing (`load_nt("Matt", ref="1-3")`), and the CLI (`aegean greek nt John 1`).
* [Recipes](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes): workflow D chains the concordance and export moves (`aegean export nt -f csv --level token`, `aegean query nt --where ins-contains-word=ἀγάπη`).
* [Greek NLP](https://github.com/ryanpavlicek/pyaegean/wiki/Greek-NLP): every tier of the pipeline, from the zero-dependency baseline to the neural model.
* [Benchmarks](https://github.com/ryanpavlicek/pyaegean/wiki/Benchmarks): the measured accuracy of each tier, including the out-of-domain NT rows.
* [Reading a Parse](https://github.com/ryanpavlicek/pyaegean/wiki/Reading-a-Parse) and [When the Tool Is Wrong](https://github.com/ryanpavlicek/pyaegean/wiki/When-the-Tool-Is-Wrong): how to read the output, and what to do when it misleads.
* [Validation & Review](https://github.com/ryanpavlicek/pyaegean/wiki/Validation-and-Review): what has and has not had outside review, and how to contribute a correction (`aegean review export` / `aegean review apply` are the CLI form of Step 6).